# 1. Data Collection with the SpaceX REST API
Goal: download past launches, enrich with rocket/launchpad/payload/core details, keep Falcon 9 only.

In [ ]:
import requests, pandas as pd, numpy as np
pd.set_option("display.max_columns",None)

In [ ]:
url="https://api.spacexdata.com/v4/launches/past"
resp=requests.get(url)
print(resp.status_code)
data=pd.json_normalize(resp.json())
data.head()

In [ ]:
data=data[["rocket","payloads","launchpad","cores","flight_number","date_utc"]]
data=data[data["cores"].map(len)==1]
data=data[data["payloads"].map(len)==1]
data["cores"]=data["cores"].map(lambda x:x[0])
data["payloads"]=data["payloads"].map(lambda x:x[0])
data["date"]=pd.to_datetime(data["date_utc"]).dt.date
data=data[data["date"]<=pd.Timestamp(2020,11,13).date()]
data.shape

In [ ]:
def get(kind,ids,field):
    return [requests.get(f"https://api.spacexdata.com/v4/{kind}/{i}").json()[field] for i in ids]
data["BoosterVersion"]=get("rockets",data["rocket"],"name")
data["PayloadMass"]=get("payloads",data["payloads"],"mass_kg")
data["Orbit"]=get("payloads",data["payloads"],"orbit")
data["LaunchSite"]=get("launchpads",data["launchpad"],"name")
data["Longitude"]=get("launchpads",data["launchpad"],"longitude")
data["Latitude"]=get("launchpads",data["launchpad"],"latitude")
data.head()

In [ ]:
data_falcon9=data[data["BoosterVersion"]=="Falcon 9"].copy()
data_falcon9["FlightNumber"]=list(range(1,data_falcon9.shape[0]+1))
data_falcon9["PayloadMass"]=data_falcon9["PayloadMass"].fillna(data_falcon9["PayloadMass"].mean())
data_falcon9.isnull().sum()

In [ ]:
data_falcon9.to_csv("../data/dataset_part_1.csv",index=False)